### 1. Objective

The objective of this notebook is to prepare the raw omniretail data for analysis while preserving all the insights and value that it holds and also documenting the reasoning behind each action.


In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel("../data/OmniRetail_Customer_Analytics_Group_Project.xlsx")

df_clean = df.copy()


In [3]:
date_cols = [
    "Snapshot_Date",
    "Signup_Date",
    "Last_Purchase_Date"
]

for col in date_cols:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        errors="coerce"
    )

### Handle Logically Invalid Values

Before addressing missing values or duplicate customers, observations
that clearly violate the logical definition of a variable are converted
to missing values.

This includes percentages outside 0–100, satisfaction scores outside
0–10, negative tenure, negative annual spend, and negative average
basket value.

Invalid values are converted to `NaN` rather than replaced with
arbitrary values because their correct value cannot be established
from the available data.

In [4]:
percentage_cols = [
    "Online_Purchase_Pct",
    "Promo_Purchase_Pct"
]

for col in percentage_cols:
    invalid = (df_clean[col] < 0) | (df_clean[col] > 100)

    df_clean.loc[invalid, col] = np.nan

invalid_satisfaction = (
    (df_clean["Satisfaction_Score"] < 0) |
    (df_clean["Satisfaction_Score"] > 10)
)

df_clean.loc[
    invalid_satisfaction,
    "Satisfaction_Score"
] = np.nan

calculated_tenure = (
    (df_clean["Snapshot_Date"].dt.year -
     df_clean["Signup_Date"].dt.year) * 12
    +
    (df_clean["Snapshot_Date"].dt.month -
     df_clean["Signup_Date"].dt.month)
    -
    (
        df_clean["Snapshot_Date"].dt.day <
        df_clean["Signup_Date"].dt.day
    ).astype(int)
)

invalid_tenure = df_clean["Tenure_Months"] < 0

df_clean.loc[
    invalid_tenure,
    "Tenure_Months"
] = calculated_tenure[invalid_tenure]

invalid_spend = df_clean["Annual_Spend_000"] < 0
invalid_basket = df_clean["Avg_Basket_Value"] < 0

df_clean.loc[
    invalid_spend,
    "Annual_Spend_000"
] = np.nan

df_clean.loc[
    invalid_basket,
    "Avg_Basket_Value"
] = np.nan

print(
    "Invalid Online Purchase %:",
    (
        (df_clean["Online_Purchase_Pct"] < 0) |
        (df_clean["Online_Purchase_Pct"] > 100)
    ).sum()
)

print(
    "Invalid Promo Purchase %:",
    (
        (df_clean["Promo_Purchase_Pct"] < 0) |
        (df_clean["Promo_Purchase_Pct"] > 100)
    ).sum()
)

print(
    "Invalid Satisfaction:",
    (
        (df_clean["Satisfaction_Score"] < 0) |
        (df_clean["Satisfaction_Score"] > 10)
    ).sum()
)

print(
    "Negative Tenure:",
    (df_clean["Tenure_Months"] < 0).sum()
)

print(
    "Negative Annual Spend:",
    (df_clean["Annual_Spend_000"] < 0).sum()
)

print(
    "Negative Basket Value:",
    (df_clean["Avg_Basket_Value"] < 0).sum()
)


Invalid Online Purchase %: 0
Invalid Promo Purchase %: 0
Invalid Satisfaction: 0
Negative Tenure: 0
Negative Annual Spend: 0
Negative Basket Value: 0


In [5]:
customer_counts = df_clean["Customer_ID"].value_counts()

duplicate_customer_ids = customer_counts[
    customer_counts > 1
].index

print(
    "Customer IDs appearing more than once:",
    len(duplicate_customer_ids)
)

print(
    "Rows belonging to duplicated customers:",
    customer_counts.loc[duplicate_customer_ids].sum()
)

duplicate_rows = df_clean[
    df_clean["Customer_ID"].isin(duplicate_customer_ids)
]

non_id_cols = [
    col for col in df_clean.columns
    if col != "Customer_ID"
]

duplicate_comparison = (
    duplicate_rows
    .groupby("Customer_ID")[non_id_cols]
    .nunique(dropna=False)
)

identical_duplicate_ids = duplicate_comparison[
    duplicate_comparison.max(axis=1) == 1
].index

conflicting_duplicate_ids = duplicate_comparison[
    duplicate_comparison.max(axis=1) > 1
].index

print("Identical duplicate IDs:", len(identical_duplicate_ids))
print("Conflicting duplicate IDs:", len(conflicting_duplicate_ids))

Customer IDs appearing more than once: 250
Rows belonging to duplicated customers: 500
Identical duplicate IDs: 150
Conflicting duplicate IDs: 100


In [6]:
before_rows = len(df_clean)

df_clean = df_clean.drop_duplicates(
    subset=non_id_cols,
    keep="first"
)

after_rows = len(df_clean)

print("Rows before:", before_rows)
print("Rows after:", after_rows)
print("Rows removed:", before_rows - after_rows)

Rows before: 5450
Rows after: 5300
Rows removed: 150


In [7]:
remaining_counts = (
    df_clean["Customer_ID"]
    .value_counts()
    .loc[lambda x: x > 1]
)

conflicting_ids = remaining_counts.index

conflicting_rows = df_clean[
    df_clean["Customer_ID"].isin(conflicting_ids)
]

duplicate_comparison = (
    conflicting_rows
    .groupby("Customer_ID")[non_id_cols]
    .nunique(dropna=False)
)

difference_summary = (
    duplicate_comparison
    .gt(1)
    .sum()
    .sort_values(ascending=False)
)

display(
    difference_summary[
        difference_summary > 0
    ]
)

Annual_Spend_000      99
Satisfaction_Score    89
dtype: int64

In [8]:
conflicting_value_counts = (
    conflicting_rows
    .groupby("Customer_ID")
    .agg(
        Spend_Valid_Count=(
            "Annual_Spend_000",
            lambda x: x.notna().sum()
        ),
        Satisfaction_Valid_Count=(
            "Satisfaction_Score",
            lambda x: x.notna().sum()
        )
    )
)

print("Annual Spend:")
print(
    conflicting_value_counts["Spend_Valid_Count"]
    .value_counts()
    .sort_index()
)

print("\nSatisfaction:")
print(
    conflicting_value_counts["Satisfaction_Valid_Count"]
    .value_counts()
    .sort_index()
)

print("Annual Spend:")
print(
    conflicting_value_counts["Spend_Valid_Count"]
    .value_counts()
    .sort_index()
)

Annual Spend:
Spend_Valid_Count
2    100
Name: count, dtype: int64

Satisfaction:
Satisfaction_Valid_Count
0     5
1     1
2    94
Name: count, dtype: int64
Annual Spend:
Spend_Valid_Count
2    100
Name: count, dtype: int64


In [9]:
conflicting_aggregated = (
    conflicting_rows
    .groupby("Customer_ID", as_index=False)
    .agg(
        Annual_Spend_000=("Annual_Spend_000", "mean"),
        Satisfaction_Score=("Satisfaction_Score", "mean")
    )
)

In [10]:
other_cols = [
    col for col in df_clean.columns
    if col not in [
        "Customer_ID",
        "Annual_Spend_000",
        "Satisfaction_Score"
    ]
]

conflicting_base = (
    conflicting_rows
    .groupby("Customer_ID", as_index=False)[other_cols]
    .first()
)

conflicting_resolved = conflicting_base.merge(
    conflicting_aggregated,
    on="Customer_ID",
    how="left"
)

df_non_conflicting = df_clean[
    ~df_clean["Customer_ID"].isin(conflicting_ids)
].copy()

df_clean = pd.concat(
    [
        df_non_conflicting,
        conflicting_resolved
    ],
    ignore_index=True
)

print("Final rows:", len(df_clean))
print(
    "Unique customers:",
    df_clean["Customer_ID"].nunique()
)



Final rows: 5200
Unique customers: 5200


### Duplicate Cleaning Summary

The duplicate-handling process reduced the dataset from 5,450 raw
records to 5,200 customer-level records.

150 exact duplicate records were removed. The remaining 100 duplicated
customers were reconciled into one record per Customer_ID using the
mean of available valid observations for Annual_Spend_000 and
Satisfaction_Score.

The final dataset contains exactly one record per customer.

### Missing-Value treatment

The next objective is to eliminate all missing values, but we need to manage missing values of different data types in different ways. Numerical, categorical, and date variables need to treated seperately.

In [11]:
missing_summary = pd.DataFrame({
    "Missing_Count": df_clean.isna().sum(),
    "Missing_%": df_clean.isna().mean() * 100
})

display(
    missing_summary[
        missing_summary["Missing_Count"] > 0
    ].sort_values(
        "Missing_%",
        ascending=False
    )
)

print(
    "Total missing cells:",
    df_clean.isna().sum().sum()
)

,Missing_Count,Missing_%
Annual_Income_000,115,2.211538
Satisfaction_Score,111,2.134615
Avg_Basket_Value,100,1.923077
Annual_Spend_000,88,1.692308
Promo_Purchase_Pct,83,1.596154
Last_Purchase_Date,73,1.403846
Age,70,1.346154
Online_Purchase_Pct,70,1.346154
Membership_Tier,54,1.038462
Region,45,0.865385


Total missing cells: 849


In [12]:
missing_by_customer = (
    df_clean.isna().sum(axis=1)
)

display(
    missing_by_customer.value_counts()
    .sort_index()
)

0    4404
1     744
2      51
3       1
Name: count, dtype: int64

In [13]:
numeric_missing_cols = [
    col for col in df_clean.select_dtypes(include=np.number).columns
    if df_clean[col].isna().any()
]

categorical_missing_cols = [
    col for col in df_clean.select_dtypes(include=["object", "category"]).columns
    if df_clean[col].isna().any()
]

date_missing_cols = [
    col for col in date_cols
    if df_clean[col].isna().any()
]

print("Numeric:", numeric_missing_cols)
print("Categorical:", categorical_missing_cols)
print("Date:", date_missing_cols)

Numeric: ['Age', 'Annual_Income_000', 'Annual_Spend_000', 'Avg_Basket_Value', 'Online_Purchase_Pct', 'Promo_Purchase_Pct', 'Satisfaction_Score']
Categorical: ['Region', 'Membership_Tier', 'Preferred_Category']
Date: ['Last_Purchase_Date']


C:\Users\singh\AppData\Local\Temp\ipykernel_18868\2475662946.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  col for col in df_clean.select_dtypes(include=["object", "category"]).columns


### Numeric Missing-Value treatment
Numeric missing values will be treated with median values. Mean is not being used because of the skewness of the data.

In [14]:
numeric_impute_cols = [
    "Annual_Income_000",
    "Satisfaction_Score",
    "Avg_Basket_Value",
    "Annual_Spend_000",
    "Promo_Purchase_Pct",
    "Age",
    "Online_Purchase_Pct"
]

imputation_values = (
    df_clean[numeric_impute_cols]
    .median()
)

display(imputation_values)

Annual_Income_000       890.0
Satisfaction_Score        7.5
Avg_Basket_Value       2878.3
Annual_Spend_000        267.9
Promo_Purchase_Pct       34.0
Age                      38.0
Online_Purchase_Pct      47.3
dtype: float64

In [15]:
for col in numeric_impute_cols:
    df_clean[col] = df_clean[col].fillna(
        imputation_values[col]
    )

### Categorical Missing-Value Treatment

Missing values in Membership_Tier, Region, and Preferred_Category are
represented using the explicit category "Unknown" rather than the mode.

This preserves the distinction between an observed category and a
category whose value was unavailable in the raw data.

In [16]:
categorical_impute_cols = [
    "Membership_Tier",
    "Region",
    "Preferred_Category"
]

for col in categorical_impute_cols:
    df_clean[col] = df_clean[col].fillna("Unknown")

## Categorical Standardization

Categorical data needs to standardized so that the data can be grouped easily and accurately.

"west", " west", "West" need to be standardized to a singular "West".

In [17]:
categorical_cols = [
    "Region",
    "City_Tier",
    "Membership_Tier",
    "Preferred_Category",
    "Acquisition_Channel"
]

for col in categorical_cols:
    print(f"\n===== {col} =====")
    print(df_clean[col].value_counts(dropna=False))


===== Region =====
Region
North      1450
South      1376
West       1237
East       1052
Unknown      45
West          4
eest          4
SOUTH         4
WEST          4
South         4
south         4
north         4
East          4
NORTH         4
Nort          4
Name: count, dtype: int64

===== City_Tier =====
City_Tier
Tier 1    2422
Tier 2    1915
Tier 3     831
T1           4
Tier-2       4
Tier1        4
Tier3        4
tier3        4
tier 1       4
T2           4
TIER 2       4
Name: count, dtype: int64

===== Membership_Tier =====
Membership_Tier
Basic        2145
Silver       1747
Gold          877
Platinum      341
Unknown        54
Platinum        4
silver          4
Glod            4
BASIC           4
basic           4
GOLD            4
SILVER          4
gold            4
platinium       4
Name: count, dtype: int64

===== Preferred_Category =====
Preferred_Category
Electronics    905
Beauty         878
Grocery        849
Home           841
Sports         837
Apparel       

Remove unnecessary white spaces first.

In [18]:
for col in categorical_cols:
    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.strip()
    )

Capitalizing properly

In [19]:
for col in categorical_cols:
    df_clean[col] = (
        df_clean[col]
        .str.title()
    )

In [20]:
df_clean["Region"] = (
    df_clean["Region"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

region_map = {
    "north": "North",
    "south": "South",
    "east": "East",
    "west": "West",
    "eest": "East",
    "nort": "North",
    "unknown": "Unknown"
}

df_clean["Region"] = df_clean["Region"].replace(region_map)

In [21]:
df_clean["City_Tier"] = (
    df_clean["City_Tier"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

city_tier_map = {
    "tier 1": "Tier 1",
    "tier1": "Tier 1",
    "t1": "Tier 1",
    "tier 2": "Tier 2",
    "tier2": "Tier 2",
    "t2": "Tier 2",
    "tier-2": "Tier 2",
    "tier 3": "Tier 3",
    "tier3": "Tier 3",
    "t3": "Tier 3"
}

df_clean["City_Tier"] = df_clean["City_Tier"].replace(city_tier_map)

In [22]:
df_clean["Membership_Tier"] = (
    df_clean["Membership_Tier"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

membership_map = {
    "basic": "Basic",
    "silver": "Silver",
    "gold": "Gold",
    "glod": "Gold",
    "platinum": "Platinum",
    "platinium": "Platinum",
    "unknown": "Unknown"
}

df_clean["Membership_Tier"] = (
    df_clean["Membership_Tier"]
    .replace(membership_map)
)

In [23]:
df_clean["Preferred_Category"] = (
    df_clean["Preferred_Category"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

preferred_category_map = {
    "electronics": "Electronics",
    "electroncs": "Electronics",
    "beauty": "Beauty",
    "grocery": "Grocery",
    "grocerry": "Grocery",
    "home": "Home",
    "sports": "Sports",
    "apparel": "Apparel",
    "unknown": "Unknown"
}

df_clean["Preferred_Category"] = (
    df_clean["Preferred_Category"]
    .replace(preferred_category_map)
)

In [24]:
df_clean["Acquisition_Channel"] = (
    df_clean["Acquisition_Channel"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

acquisition_map = {
    "organic": "Organic",
    "store walk-in": "Store Walk-in",
    "store walkin": "Store Walk-in",
    "search ads": "Search Ads",
    "social media": "Social Media",
    "socialmedia": "Social Media",
    "referral": "Referral",
    "referrral": "Referral",
    "marketplace": "Marketplace",
    "market place": "Marketplace"
}

df_clean["Acquisition_Channel"] = (
    df_clean["Acquisition_Channel"]
    .replace(acquisition_map)
)

### Categorical Standardization Result

Categorical values were standardized using explicit mappings for
capitalization differences, whitespace variations, abbreviations,
alternative formatting, and identified spelling errors.

The standardized dataset now uses a single canonical representation
for each valid category. Missing categorical values remain represented
as "Unknown".

Validation confirmed that no unexpected category labels remain.

In [25]:
invalid_purchase_sequence = (
    df_clean["Last_Purchase_Date"] <
    df_clean["Signup_Date"]
)

print(
    "Last purchase before signup:",
    invalid_purchase_sequence.sum()
)

Last purchase before signup: 4


In [26]:
display(
    df_clean.loc[
        invalid_purchase_sequence,
        [
            "Customer_ID",
            "Signup_Date",
            "Last_Purchase_Date",
            "Snapshot_Date",
            "Annual_Spend_000",
            "Store_Visits_Year",
            "Online_Orders_Year"
        ]
    ]
)

,Customer_ID,Signup_Date,Last_Purchase_Date,Snapshot_Date,Annual_Spend_000,Store_Visits_Year,Online_Orders_Year
1547,C100548,2026-07-02,2026-06-14,2026-08-01,278.0,19,30
1923,C103972,2026-07-02,2026-06-27,2026-08-01,353.1,18,26
3765,C102098,2026-07-02,2026-06-14,2026-08-01,98.7,12,21
3806,C104187,2026-07-02,2026-06-26,2026-08-01,217.4,27,6


In [27]:
df_clean.loc[
    invalid_purchase_sequence,
    "Last_Purchase_Date"
] = pd.NaT

### Date Cleaning Result

The three date variables were validated against the common snapshot
date of 2026-08-01.

No signup dates or purchase dates occur after the snapshot date.
Four records contained an invalid sequence in which the last purchase
preceded signup. Because the correct date could not be established,
these purchase dates were converted to missing.

The final dataset therefore retains only chronologically defensible
purchase dates.

## Export Clean Dataset

The validated customer-level dataset is exported as the primary input
for exploratory data analysis, feature engineering, PCA, clustering,
and subsequent customer segmentation.

In [28]:
output_path = "../data/OmniRetail_Cleaned.xlsx"

df_clean.to_excel(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: ../data/OmniRetail_Cleaned.xlsx


## Cleaning Summary

The raw OmniRetail dataset contained 5,450 records representing
5,200 unique customers.

The following cleaning operations were performed:

1. Standardized date data types.
2. Converted logically invalid percentage, satisfaction, and financial
   values to missing.
3. Recalculated four invalid tenure values using signup and snapshot
   dates.
4. Removed 150 exact duplicate records.
5. Reconciled 100 customers with conflicting duplicate records by
   aggregating valid customer-level observations.
6. Median-imputed low-level numeric missingness.
7. Represented missing categorical values using "Unknown".
8. Standardized inconsistent categorical labels.
9. Converted four chronologically invalid purchase dates to missing.
10. Validated the resulting dataset across uniqueness, numeric ranges,
    categorical values, and date logic.

The resulting dataset contains 5,200 customer-level records and is
ready for exploratory data analysis.